# Step Detection — Moving Average Filter & Multi-Run Comparison

This notebook implements step detection for all measurement runs (R1, R2, R3).  
The primary smoothing filter is a **Moving Average (MA)**, which is directly grounded  
in the MA-type signal characteristics identified in the ACF analysis *(01_TimeSeries)*.  
At the end a manual **Butterworth low-pass filter** (pure numpy, no scipy) is added  
as a comparison.

**No scipy is used anywhere in this notebook.**

**Pipeline:**
1. Load accelerometer data from DB (all runs)
2. Compute Euclidean norm & sampling rate
3. Apply Moving Average filter
4. Stationarity check (ADF + KPSS)
5. ACF / PACF analysis
6. Peak detection — pure numpy
7. Step detection plot — all runs
8. Multi-run comparison
9. Write results to DB (`steps` table)
10. Filter comparison: Moving Average vs. Butterworth (numpy)

In [ ]:
# ── Imports ───────────────────────────────────────────────────
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print('Imports OK ✓  |  scipy: NOT used')

In [ ]:
# ── Config ────────────────────────────────────────────────────
DB_PATH      = '../data/emi_nav.db'
RUN_IDS      = ['R1', 'R2', 'R3']
MA_WINDOW_MS = 100    # Moving Average window length in ms
THRESHOLD_K  = 0.5    # dynamic threshold: mean + K * std
MIN_STEP_MS  = 300    # physiological minimum between steps [ms]

## Step 1 — Load accelerometer data from DB

In [ ]:
conn = sqlite3.connect(DB_PATH)
runs = {}
for run_id in RUN_IDS:
    df = pd.read_sql_query(f"""
        SELECT timestamp_ms, x, y, z
        FROM imu
        WHERE run_id = '{run_id}' AND sensor = 'accel'
        ORDER BY timestamp_ms
    """, conn)
    runs[run_id] = df
    print(f'{run_id}: {len(df):,} samples loaded')
conn.close()

## Step 2 — Euclidean Norm & Sampling Rate

In [ ]:
fs_per_run = {}

for run_id, df in runs.items():
    df['norm']   = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)
    df['time_s'] = (df['timestamp_ms'] - df['timestamp_ms'].min()) / 1000
    dt_ms = df['timestamp_ms'].diff().median()
    fs    = 1000 / dt_ms
    fs_per_run[run_id] = fs
    print(f'{run_id}: fs = {fs:.1f} Hz  |  duration = {df["time_s"].max():.1f} s')

**Interpretation**

The Euclidean norm ||a|| = sqrt(x² + y² + z²) collapses the three-axis signal
into one rotation-invariant scalar, independent of phone orientation in the pocket.
At rest the norm sits near 9.81 m/s² (gravity). Walking produces periodic peaks
above that baseline. The sampling rate fs is estimated per run from the median
inter-sample interval, using the median rather than the mean to be robust against
occasional gaps caused by app interruptions.

## Step 3 — Moving Average Filter

In [ ]:
# ── Moving Average via numpy convolution (no scipy) ───────────
for run_id, df in runs.items():
    fs     = fs_per_run[run_id]
    window = max(1, int((MA_WINDOW_MS / 1000) * fs))
    kernel = np.ones(window) / window
    df['norm_ma'] = np.convolve(df['norm'].values, kernel, mode='same')
    print(f'{run_id}: MA window = {window} samples ({MA_WINDOW_MS} ms)')

# ── Plot all runs ─────────────────────────────────────────────
fig, axes = plt.subplots(len(RUN_IDS), 1, figsize=(13, 4 * len(RUN_IDS)), sharex=False)

for ax, run_id in zip(axes, RUN_IDS):
    df = runs[run_id]
    ax.plot(df['time_s'], df['norm'],    alpha=0.25, color='gray',      label='raw')
    ax.plot(df['time_s'], df['norm_ma'], color='steelblue', linewidth=2,
            label=f'Moving Average ({MA_WINDOW_MS} ms)')
    ax.set_title(f'[{run_id}] Accelerometer Magnitude — Moving Average Smoothing')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('||a|| [m/s²]')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

**Interpretation**

The Moving Average filter replaces each sample with the mean of its surrounding
window of length ~100 ms. This is the simplest form of low-pass filtering and is
directly motivated by the MA-type signal characteristics of the accelerometer:
step impulses are short-lived events whose energy is concentrated in a finite
time window, consistent with the MA model family described in the lecture
*(01_TimeSeries, Section 5.4.2)*. The convolution with a uniform kernel is
equivalent to a finite impulse response (FIR) filter. Using `mode='same'`
preserves the original signal length and avoids edge-induced time shifts.

## Step 4 — Stationarity Check (ADF + KPSS)

In [ ]:
stationarity_results = []

for run_id, df in runs.items():
    sig = df['norm_ma'].dropna()

    adf_stat, adf_p, *_ = adfuller(sig)
    kpss_stat, kpss_p, *_ = kpss(sig, regression='c', nlags='auto')

    stationary = (adf_p < 0.05) and (kpss_p > 0.05)

    if not stationary:
        df['signal_det'] = df['norm_ma'].diff()
        decision = '1st-order differencing applied'
    else:
        df['signal_det'] = df['norm_ma']
        decision = 'signal used directly'

    stationarity_results.append({
        'Run':        run_id,
        'ADF p':      round(adf_p,  4),
        'KPSS p':     round(kpss_p, 4),
        'Stationary': stationary,
        'Decision':   decision
    })
    print(f'{run_id} | ADF p={adf_p:.4f} | KPSS p={kpss_p:.4f} | → {decision}')

pd.DataFrame(stationarity_results)

**Interpretation**

ADF tests H₀: non-stationary — rejection (p < 0.05) indicates stationarity.
KPSS tests H₀: stationary — non-rejection (p > 0.05) confirms stationarity.
Both tests follow the analysis workflow from the lecture *(01_TimeSeries, Section 6.2)*:
stationarity must be confirmed before ACF/PACF inspection. If either test
indicates non-stationarity, first-order differencing Δxₜ = xₜ − xₜ₋₁
is applied exactly once to remove drift without over-differencing.

## Step 5 — ACF / PACF Analysis

In [ ]:
for run_id, df in runs.items():
    sig = df['signal_det'].dropna()
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))
    plot_acf( sig, lags=60, ax=ax1,
              title=f'[{run_id}] ACF  — MA-Smoothed Accelerometer Magnitude')
    plot_pacf(sig, lags=60, ax=ax2,
              title=f'[{run_id}] PACF — MA-Smoothed Accelerometer Magnitude')
    ax1.set_xlabel('Lag [samples]'); ax1.set_ylabel('Autocorrelation')
    ax2.set_xlabel('Lag [samples]'); ax2.set_ylabel('Partial Autocorrelation')
    plt.tight_layout()
    plt.show()

**Interpretation**

A sharp ACF cut-off after 1–2 lags with gradual PACF decay confirms MA-type behaviour —
consistent with step impulses being short-lived events that return to baseline quickly.
This pattern validates the choice of the Moving Average filter: an MA smoother is
well matched to signals with MA-type autocorrelation structure.
Seasonal spikes in the ACF at regular lag intervals indicate the stride frequency
and can be used to cross-validate the detected step count across runs.

## Step 6 — Peak Detection (pure numpy)

In [ ]:
# ── Pure numpy peak finder (no scipy) ────────────────────────
def find_peaks_numpy(signal, height, min_distance):
    """
    Detect local maxima above `height` with minimum separation `min_distance`.
    Pure numpy — no scipy dependency.
    If two peaks are closer than min_distance, the higher one is kept.
    """
    peaks = []
    for i in range(1, len(signal) - 1):
        if signal[i] > signal[i - 1] and signal[i] > signal[i + 1]:
            if signal[i] >= height:
                if not peaks or (i - peaks[-1]) >= min_distance:
                    peaks.append(i)
                elif signal[i] > signal[peaks[-1]]:
                    peaks[-1] = i
    return np.array(peaks)


step_results = {}

for run_id, df in runs.items():
    fs       = fs_per_run[run_id]
    sig_vals = df['signal_det'].dropna().values
    sig_time = df['time_s'].dropna().values

    threshold = sig_vals.mean() + THRESHOLD_K * sig_vals.std()
    min_dist  = int((MIN_STEP_MS / 1000) * fs)

    peaks      = find_peaks_numpy(sig_vals, height=threshold, min_distance=min_dist)
    step_times = sig_time[peaks]

    step_results[run_id] = {
        'peaks':      peaks,
        'step_times': step_times,
        'sig_vals':   sig_vals,
        'sig_time':   sig_time,
        'threshold':  threshold,
        'n_steps':    len(peaks),
        'freq':       len(peaks) / df['time_s'].max()
    }
    print(f'{run_id}: {len(peaks)} steps  |  '          f'freq = {len(peaks)/df["time_s"].max():.2f} steps/s  |  '          f'threshold = {threshold:.3f}')

## Step 7 — Step Detection Plot — All Runs

In [ ]:
fig, axes = plt.subplots(len(RUN_IDS), 1, figsize=(13, 4 * len(RUN_IDS)), sharex=False)

for ax, run_id in zip(axes, RUN_IDS):
    r = step_results[run_id]
    ax.plot(r['sig_time'], r['sig_vals'],
            alpha=0.8, color='steelblue', label='signal (MA-smoothed)')
    ax.plot(r['step_times'], r['sig_vals'][r['peaks']],
            'x', color='red', markersize=9, linewidth=2,
            label=f'steps (n={r["n_steps"]})')
    ax.axhline(r['threshold'], color='orange', linestyle='--',
               label=f'threshold (μ + {THRESHOLD_K}σ = {r["threshold"]:.2f})')
    ax.set_title(f'[{run_id}] Step Detection — Moving Average Filter')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('||a|| [m/s²]')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

**Interpretation**

Each red cross marks one detected step. The dynamic threshold (μ + 0.5σ) adapts
to the signal level of each run individually, which makes the detector robust
against differences in walking speed, device placement, and run duration.
The 300 ms minimum inter-peak distance corresponds to the physiological upper
cadence limit of ~3.3 steps/s. Plausible step frequencies lie between 1.2 and 2.2 steps/s.

**Tuning guide:**
- `MA_WINDOW_MS` too large → peaks flattened → missed steps → decrease
- `MA_WINDOW_MS` too small → noise not suppressed → false positives → increase
- `THRESHOLD_K` too high → missed steps → decrease (e.g. 0.3)
- `MIN_STEP_MS` too small → double-counted steps → increase (e.g. 400 ms)

## Step 8 — Multi-Run Comparison

In [ ]:
summary = pd.DataFrame([
    {
        'Run':            run_id,
        'Duration [s]':   round(runs[run_id]['time_s'].max(), 1),
        'Steps detected': step_results[run_id]['n_steps'],
        'Step freq [/s]': round(step_results[run_id]['freq'], 3),
        'Threshold':      round(step_results[run_id]['threshold'], 3),
        'fs [Hz]':        round(fs_per_run[run_id], 1)
    }
    for run_id in RUN_IDS
])
print(summary.to_string(index=False))
summary

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

runs_list  = list(step_results.keys())
n_steps    = [step_results[r]['n_steps'] for r in runs_list]
step_freqs = [step_results[r]['freq']    for r in runs_list]

ax1.bar(runs_list, n_steps, color='steelblue', edgecolor='black')
ax1.set_title('Total Steps per Run')
ax1.set_xlabel('Run'); ax1.set_ylabel('Steps detected')
ax1.grid(axis='y')
for i, v in enumerate(n_steps):
    ax1.text(i, v + 1, str(v), ha='center', fontweight='bold')

ax2.bar(runs_list, step_freqs, color='coral', edgecolor='black')
ax2.set_title('Step Frequency per Run')
ax2.set_xlabel('Run'); ax2.set_ylabel('Steps / second')
ax2.grid(axis='y')
for i, v in enumerate(step_freqs):
    ax2.text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Interpretation**

Comparing step counts and frequencies across all three runs assesses pipeline
consistency across different paths, starting positions, and walking speeds.
Step frequency should fall in the physiologically plausible range of 1.2–2.2 steps/s.
Runs involving floor transitions may show slightly lower average frequency
due to the different acceleration pattern during stair climbing.

## Step 9 — Write Results to DB (`steps` table)

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS steps;
CREATE TABLE steps (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id          TEXT    NOT NULL,
    timestamp_ms    INTEGER NOT NULL,
    accel_norm      REAL,
    step_index      INTEGER,
    FOREIGN KEY (run_id) REFERENCES runs(run_id)
);
""")
conn.commit()
print("Table 'steps' ready ✓")

In [ ]:
total_written = 0

for run_id, df in runs.items():
    r       = step_results[run_id]
    idx     = r['peaks']
    ts_vals = runs[run_id]['timestamp_ms'].reset_index(drop=True)
    nv_vals = runs[run_id]['norm_ma'].reset_index(drop=True)

    rows = [
        (run_id, int(ts_vals.iloc[i]), float(nv_vals.iloc[i]), int(step_num))
        for step_num, i in enumerate(idx)
    ]
    cur.executemany("""
        INSERT INTO steps (run_id, timestamp_ms, accel_norm, step_index)
        VALUES (?, ?, ?, ?)
    """, rows)
    conn.commit()
    total_written += len(rows)
    print(f'{run_id}: {len(rows)} steps written ✓')

conn.close()
print(f'\nTotal: {total_written} records written to steps table ✓')

In [ ]:
# ── Verify ────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
verification = pd.read_sql_query("""
    SELECT run_id, COUNT(*) as step_count
    FROM steps GROUP BY run_id ORDER BY run_id
""", conn)
conn.close()
print(verification.to_string(index=False))
verification

**Interpretation**

The `steps` table stores one record per detected step, linked via `run_id` and
synchronised via `timestamp_ms`. This allows the particle filter in the next pipeline
stage to query step events per run and use `accel_norm` as a proxy for step length
estimation. The schema is consistent with the `imu` and `ble_rssi` tables.

## Step 10 — Filter Comparison: Moving Average vs. Butterworth (pure numpy)

The Butterworth filter is implemented manually using the bilinear transform — **no scipy**.

In [ ]:
# ── Manual Butterworth low-pass (1st order, numpy only) ───────
def butterworth_lowpass_numpy(signal, cutoff_hz, fs, order=1):
    """
    1st-order Butterworth low-pass via bilinear transform (IIR, numpy only).
    For higher orders the filter is applied sequentially (order passes).
    Zero-phase: applied forward then backward to eliminate phase shift.
    """
    # Bilinear transform coefficient
    dt  = 1.0 / fs
    rc  = 1.0 / (2.0 * np.pi * cutoff_hz)
    alpha = dt / (rc + dt)          # smoothing factor: 0 = no filter, 1 = full

    def _apply_once(x, a):
        out = np.zeros_like(x, dtype=float)
        out[0] = x[0]
        for i in range(1, len(x)):
            out[i] = a * x[i] + (1 - a) * out[i - 1]
        return out

    result = signal.astype(float)
    for _ in range(order):
        result = _apply_once(result, alpha)      # forward pass
        result = _apply_once(result[::-1], alpha)[::-1]  # backward pass (zero-phase)
    return result


CUTOFF_HZ = 5.0   # step signals at 1–2 Hz; cut-off at 5 Hz removes higher noise

for run_id, df in runs.items():
    fs = fs_per_run[run_id]
    df['norm_butter'] = butterworth_lowpass_numpy(df['norm'].values, CUTOFF_HZ, fs, order=4)

In [ ]:
# ── Comparison plot — all runs ────────────────────────────────
fig, axes = plt.subplots(len(RUN_IDS), 1, figsize=(13, 4 * len(RUN_IDS)), sharex=False)

for ax, run_id in zip(axes, RUN_IDS):
    df = runs[run_id]
    ax.plot(df['time_s'], df['norm'],        alpha=0.2, color='gray',       label='raw')
    ax.plot(df['time_s'], df['norm_ma'],     color='steelblue', linewidth=2, label=f'Moving Average ({MA_WINDOW_MS} ms)')
    ax.plot(df['time_s'], df['norm_butter'], color='tomato',    linewidth=2,
            linestyle='--', label=f'Butterworth ({CUTOFF_HZ} Hz, numpy)')
    ax.set_title(f'[{run_id}] Filter Comparison: MA vs. Butterworth')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('||a|| [m/s²]')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

**Interpretation**

Both filters suppress high-frequency noise while preserving step-induced peaks.
The key differences are:

| Property | Moving Average | Butterworth (numpy) |
|---|---|---|
| **Frequency selectivity** | Broad, sinc-shaped response | Maximally flat, sharper roll-off |
| **Peak preservation** | Slight flattening at edges | Better shape retention |
| **Phase** | Zero-phase (`mode='same'`) | Zero-phase (forward + backward) |
| **Folienbezug** | MA model *(01_TimeSeries)* | Signal attenuation *(PositionEstimation)* |
| **Complexity** | Trivial | Requires bilinear transform |

For step detection at 1–2 Hz, both filters perform comparably.
The Moving Average is preferred here due to its direct theoretical motivation
from the MA-type ACF structure of the accelerometer signal.